In [1]:
# =====================================================
# 05 - Risk Scoring
# =====================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
project_path = "/content/drive/MyDrive/Mobile_API_Misuse_Detector/"
iso_df = pd.read_csv(
    project_path + "data/processed/isolation_forest_results.csv"
)

dbscan_df = pd.read_csv(
    project_path + "data/processed/dbscan_results.csv"
)

ae_df = pd.read_csv(
    project_path + "data/processed/autoencoder_results.csv"
)


risk_df = iso_df.copy()

risk_df["dbscan_label"] = dbscan_df["dbscan_label"]

risk_df["ae_prediction"] = ae_df["ae_prediction"]

risk_df["ae_reconstruction_error"] = ae_df[
    "ae_reconstruction_error"
]

risk_df["iso_anomaly_score"] = np.where(
    risk_df["iso_prediction"] == -1,
    1,
    0
)

risk_df["dbscan_anomaly_score"] = np.where(
    risk_df["dbscan_label"] == -1,
    1,
    0
)

risk_df["ae_anomaly_score"] = np.where(
    risk_df["ae_prediction"] == -1,
    1,
    0
)

risk_df["risk_score"] = (
    risk_df["iso_anomaly_score"] * 0.4 +
    risk_df["dbscan_anomaly_score"] * 0.3 +
    risk_df["ae_anomaly_score"] * 0.3
)


def classify_risk(score):

    if score >= 0.8:
        return "Critical"

    elif score >= 0.5:
        return "High"

    elif score >= 0.3:
        return "Medium"

    else:
        return "Low"


risk_df["risk_level"] = risk_df["risk_score"].apply(
    classify_risk
)


risk_df[
    [
        "ip",
        "risk_score",
        "risk_level",
        "iso_prediction",
        "dbscan_label",
        "ae_prediction"
    ]
].sort_values(
    by="risk_score",
    ascending=False
).head(20)

risk_df["risk_level"].value_counts()


Mounted at /content/drive


,count
risk_level,
Low,2114
Medium,23
High,22
Critical,14


In [2]:
risk_df.to_csv(
    project_path + "data/processed/final_risk_scores.csv",
    index=False
)

print("Risk scoring terminé.")

Risk scoring terminé.
